<a href="https://colab.research.google.com/github/NicoleKundiona/PPKL-proteins-from-toxoplasma-gondii/blob/main/Demonstration_BioInfo_for_Nicole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install biopython

#Convert from ToxoDB ID to clean ID

In [ ]:
from Bio import SeqIO
import pandas as pd
import re

def extract_tgg1_gene_id(seq_id):
    match = re.search(r"(TGGT1_\d+)", seq_id)
    return match.group(1) if match else None


fasta_file = "/content/drive/MyDrive/Colab Notebooks/tgg1/boolean_question_TranscriptRecordClasses_TranscriptRecordClass.fasta"

rows = []

for record in SeqIO.parse(fasta_file, "fasta"):
    original_id = record.id
    clean_gene_id = extract_tgg1_gene_id(original_id)

    rows.append({
        "original_id": original_id,
        "clean_gene_id": clean_gene_id,
        "length": len(record.seq),
        "sequence": str(record.seq),
    })

df = pd.DataFrame(rows)
df.to_csv("/content/drive/MyDrive/Colab Notebooks/tgg1/cleaned_tgg1_ids.csv", index=False)

print(df.head())

          original_id clean_gene_id  length  \
0  TGGT1_215460-t26_1  TGGT1_215460     267   
1  TGGT1_240660-t26_1  TGGT1_240660     483   
2  TGGT1_248290-t26_1  TGGT1_248290    3345   
3  TGGT1_257780-t26_1  TGGT1_257780    1698   
4  TGGT1_259840-t26_1  TGGT1_259840    2091   

                                            sequence  
0  MERSKQSSRFFLCMQRPPPHSAHTPRGDFPVYAANNRMHARLFIRP...  
1  MHKVLDQSRKSPDVSKFKDHQHSSSISRLKPFASDTGLRRFRSSIS...  
2  MFVERLLHVHHGGLPITSLDFQPHAAPWNMERLATAAANCVKIWVL...  
3  MRSPFGPLRGWRRDLGSVLLSSDCPVVSPRPTSDGGFPTLLPGPSG...  
4  MVGQDFYYLCRHPNVFRVPTRDEPLLTHALPLEILDPPEAAPFSKR...  


#Extract from NCBI

In [ ]:
from Bio import Entrez
import pandas as pd
import time

# -----------------------------
# Settings
# -----------------------------
Entrez.email = "nickykundy@gamil.com"  # Please change this to your email

input_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/cleaned_tgg1_ids.csv"
output_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_by_ncbi.csv"

id_column = "clean_gene_id"  # change if your column name is different


unknown_keywords = [
    "hypothetical",
    "uncharacterized",
    "unknown",
    "unknown function",
    "protein of unknown function",
    "predicted protein",
]


# -----------------------------
# Helper functions
# -----------------------------
def is_unknown_title(title):
    title = str(title).lower()
    return any(k in title for k in unknown_keywords)


def classify_product(title):
    title = str(title).strip()

    if title == "":
        return "not_found"

    if is_unknown_title(title):
        return "unknown_hypothetical"

    return "known"


def choose_best_ncbi_hit(clean_id, summaries):
    """
    Choose one best NCBI protein record from multiple hits.
    """

    candidates = []

    for s in summaries:
        accession = s.get("AccessionVersion", "")
        title = s.get("Title", "")
        organism = s.get("Organism", "")
        uid = s.get("Id", "")

        score = 0

        # Exact TGME49 ID in the title is very useful
        if clean_id in title:
            score += 100

        # Toxoplasma gondii match
        if "toxoplasma gondii" in organism.lower():
            score += 50
        elif "toxoplasma gondii" in title.lower():
            score += 30

        # RefSeq-like accession is often preferable
        if accession.startswith(("XP_", "NP_", "YP_", "WP_")):
            score += 20

        # Named protein is more informative than hypothetical/uncharacterized
        if not is_unknown_title(title):
            score += 10

        candidates.append({
            "score": score,
            "accession": accession,
            "title": title,
            "organism": organism,
            "uid": uid,
        })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    return candidates[0], candidates


def search_ncbi_protein_best(clean_id, retmax=10):
    """
    Search NCBI Protein using clean TGG1 ID.
    If multiple hits are found, choose one best hit.
    """

    term = f'"{clean_id}" AND "Toxoplasma gondii"[Organism]'

    handle = Entrez.esearch(
        db="protein",
        term=term,
        retmax=retmax
    )
    record = Entrez.read(handle)
    handle.close()

    ids = record["IdList"]

    if len(ids) == 0:
        return None, []

    handle = Entrez.esummary(
        db="protein",
        id=",".join(ids)
    )
    summaries = Entrez.read(handle)
    handle.close()

    best, all_candidates = choose_best_ncbi_hit(clean_id, summaries)

    return best, all_candidates


# -----------------------------
# Main
# -----------------------------
df = pd.read_csv(input_csv)

if id_column not in df.columns:
    raise ValueError(
        f"Column '{id_column}' was not found. "
        f"Available columns are: {list(df.columns)}"
    )

results = []

unique_ids = df[id_column].dropna().astype(str).unique()

for i, clean_id in enumerate(unique_ids, start=1):
    print(f"[{i}/{len(unique_ids)}] Searching {clean_id}")

    try:
        best, candidates = search_ncbi_protein_best(clean_id, retmax=10)
    except Exception as e:
        print(f"  Failed: {clean_id} | {e}")
        best = None
        candidates = []

    if best is None:
        row = {
            id_column: clean_id,
            "ncbi_accession": "",
            "protein_title": "",
            "organism": "",
            "ncbi_uid": "",
            "best_hit_score": "",
            "all_candidate_accessions": "",
            "all_candidate_titles": "",
            "annotation_source": "not_found_in_ncbi",
            "category": "not_found",
        }
    else:
        protein_title = best["title"]

        row = {
            id_column: clean_id,
            "ncbi_accession": best["accession"],
            "protein_title": protein_title,
            "organism": best["organism"],
            "ncbi_uid": best["uid"],
            "best_hit_score": best["score"],
            "all_candidate_accessions": "; ".join([c["accession"] for c in candidates]),
            "all_candidate_titles": " || ".join([c["title"] for c in candidates]),
            "annotation_source": "ncbi",
            "category": classify_product(protein_title),
        }

    results.append(row)

    # Be gentle to NCBI server
    time.sleep(0.35)

anno = pd.DataFrame(results)

out = df.merge(anno, on=id_column, how="left")

out.to_csv(output_csv, index=False)

print("\nClassification summary:")
print(out["category"].value_counts(dropna=False))

print(f"\nSaved: {output_csv}")

[1/476] Searching TGGT1_215460
[2/476] Searching TGGT1_240660
[3/476] Searching TGGT1_248290
[4/476] Searching TGGT1_257780
[5/476] Searching TGGT1_259840
[6/476] Searching TGGT1_260600
[7/476] Searching TGGT1_270270
[8/476] Searching TGGT1_280390
[9/476] Searching TGGT1_285660
[10/476] Searching TGGT1_290630
[11/476] Searching TGGT1_293900
[12/476] Searching TGGT1_297730
[13/476] Searching TGGT1_203620
[14/476] Searching TGGT1_217520
[15/476] Searching TGGT1_223790
[16/476] Searching TGGT1_246920
[17/476] Searching TGGT1_275630
[18/476] Searching TGGT1_277840
[19/476] Searching TGGT1_281430
[20/476] Searching TGGT1_306220
[21/476] Searching TGGT1_202870
[22/476] Searching TGGT1_208050
[23/476] Searching TGGT1_227948
[24/476] Searching TGGT1_242020
[25/476] Searching TGGT1_259940
[26/476] Searching TGGT1_266310
[27/476] Searching TGGT1_312580
[28/476] Searching TGGT1_208400
[29/476] Searching TGGT1_224600
[30/476] Searching TGGT1_244700
[31/476] Searching TGGT1_289490
[32/476] Searchin

#Extract from Uniport

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path

# -----------------------------
# Settings
# -----------------------------
input_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_by_ncbi.csv"
output_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_tgg1_proteins.csv"

id_column = "clean_gene_id"   # change this if your column name is different

unknown_keywords = [
    "hypothetical",
    "uncharacterized",
    "unknown",
    "unknown function",
    "protein of unknown function",
    "predicted protein",
]

# Words that often indicate named/known function.
# This list is not required, but useful for confidence labels.
known_keywords = [
    "kinase",
    "phosphatase",
    "transporter",
    "transferase",
    "synthase",
    "synthetase",
    "dehydrogenase",
    "oxidase",
    "reductase",
    "protease",
    "peptidase",
    "helicase",
    "polymerase",
    "ribosomal",
    "histone",
    "actin",
    "tubulin",
    "myosin",
    "chaperone",
    "heat shock",
    "dna-binding",
    "rna-binding",
    "transcription factor",
    "surface antigen",
    "microneme",
    "rhoptry",
    "dense granule",
]

def query_uniprot_by_tggt1(clean_id):
    """
    Search UniProtKB by TGME49 gene-style ID.
    Returns the first matched record as a dictionary, or None.
    """

    url = "https://rest.uniprot.org/uniprotkb/search"

    # Broad phrase search works better than gene: for ToxoDB-style IDs.
    query = f'"{clean_id}" AND taxonomy_id:5811'

    params = {
        "query": query,
        "format": "tsv",
        "fields": "accession,id,protein_name,gene_names,organism_name,xref_toxodb,length",
        "size": 1,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    text = r.text.strip()

    if not text:
        return None

    lines = text.split("\n")

    if len(lines) < 2:
        return None

    header = lines[0].split("\t")
    values = lines[1].split("\t")

    return dict(zip(header, values))


def classify_from_name(protein_name):
    """
    Two-category classification:
    - unknown_hypothetical
    - known
    """

    name = str(protein_name).strip()
    name_lower = name.lower()

    if name == "":
        return "unknown_hypothetical"

    if any(k in name_lower for k in unknown_keywords):
        return "unknown_hypothetical"

    return "known"


# -----------------------------
# Main
# -----------------------------
df = pd.read_csv(input_csv)

if id_column not in df.columns:
    raise ValueError(f"Column '{id_column}' was not found. Available columns: {list(df.columns)}")

results = []

unique_ids = df[id_column].dropna().astype(str).unique()

for i, clean_id in enumerate(unique_ids, start=1):
    print(f"[{i}/{len(unique_ids)}] Searching {clean_id}")

    try:
        result = query_uniprot_by_tgme49(clean_id)
    except Exception as e:
        print(f"  Failed: {clean_id} | {e}")
        result = None

    if result is None:
        row = {
            id_column: clean_id,
            "uniprot_accession": "",
            "uniprot_entry": "",
            "protein_name": "",
            "gene_names": "",
            "organism": "",
            "toxodb_xref": "",
            "uniprot_length": "",
            "annotation_source": "not_found_in_uniprot",
            "category": "unknown_hypothetical",
        }
    else:
        protein_name = result.get("Protein names", "")

        row = {
            id_column: clean_id,
            "uniprot_accession": result.get("Entry", ""),
            "uniprot_entry": result.get("Entry Name", ""),
            "protein_name": protein_name,
            "gene_names": result.get("Gene Names", ""),
            "organism": result.get("Organism", ""),
            "toxodb_xref": result.get("ToxoDB", ""),
            "uniprot_length": result.get("Length", ""),
            "annotation_source": "uniprot",
            "category": classify_from_name(protein_name),
        }

    results.append(row)

    # Be gentle to the server
    time.sleep(0.2)

anno = pd.DataFrame(results)

# Merge annotation back to the original CSV
out = df.merge(anno, on=id_column, how="left")

out.to_csv(output_csv, index=False)

print("\nClassification summary:")
print(out["category_y"].value_counts(dropna=False))

print(f"\nSaved: {output_csv}")

[1/476] Searching TGGT1_215460
  Failed: TGGT1_215460 | name 'query_uniprot_by_tgme49' is not defined
[2/476] Searching TGGT1_240660
  Failed: TGGT1_240660 | name 'query_uniprot_by_tgme49' is not defined
[3/476] Searching TGGT1_248290
  Failed: TGGT1_248290 | name 'query_uniprot_by_tgme49' is not defined
[4/476] Searching TGGT1_257780
  Failed: TGGT1_257780 | name 'query_uniprot_by_tgme49' is not defined
[5/476] Searching TGGT1_259840
  Failed: TGGT1_259840 | name 'query_uniprot_by_tgme49' is not defined
[6/476] Searching TGGT1_260600
  Failed: TGGT1_260600 | name 'query_uniprot_by_tgme49' is not defined
[7/476] Searching TGGT1_270270
  Failed: TGGT1_270270 | name 'query_uniprot_by_tgme49' is not defined
[8/476] Searching TGGT1_280390
  Failed: TGGT1_280390 | name 'query_uniprot_by_tgme49' is not defined
[9/476] Searching TGGT1_285660
  Failed: TGGT1_285660 | name 'query_uniprot_by_tgme49' is not defined
[10/476] Searching TGGT1_290630
  Failed: TGGT1_290630 | name 'query_uniprot_by_tg

### Extracting Unknown/Hypothetical Protein Sequences

Based on the UniProt classification (`category_y`), we can filter the `out` DataFrame to identify and display the sequences of proteins that are still categorized as 'unknown_hypothetical'.

In [ ]:
import pandas as pd

# Load the classified proteins DataFrame
output_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_tgg1_proteins.csv"
out = pd.read_csv(output_csv)

if 'category_y' in out.columns:
    classification_column = 'category_y'
    print("Using UniProt classification ('category_y').")
elif 'category_x' in out.columns:
    classification_column = 'category_x'
    print("Warning: 'category_y' not found. Falling back to NCBI classification ('category_x').")
else:
    raise KeyError("Neither 'category_y' nor 'category_x' found in the DataFrame. Cannot classify proteins.")

# Filter for unknown/hypothetical proteins based on the selected classification column
unknown_proteins_df = out[out[classification_column] == 'unknown_hypothetical']

print(f"Number of unknown/hypothetical proteins: {len(unknown_proteins_df)}")
print("\nExample sequences of unknown/hypothetical proteins:")

# Display the sequences of these proteins
# You might want to adjust how many are displayed, e.g., using .head() or iterating
for index, row in unknown_proteins_df.head(5).iterrows():
    print(f"ID: {row['clean_gene_id']}\nSequence: {row['sequence']}\n")

Using UniProt classification ('category_y').
Number of unknown/hypothetical proteins: 478

Example sequences of unknown/hypothetical proteins:
ID: TGGT1_215460
Sequence: MERSKQSSRFFLCMQRPPPHSAHTPRGDFPVYAANNRMHARLFIRPSRLGSTLYFPAWQEERLRIRLQSAPSAEVKDPETFPSRFFRPSKRTTDGCTEWGTEAHSGFLSRHLLPRPGLLSPRSPLFSVTESNMADSGAFSLRFRKLKTNPLLQRKQFGVDILHPSRGSVSKKELVEKIAKQFRVSDSQSIVVFGLKTAFGGGRSSGFAMIYDNPNAAKKFENRFRLVRLGYVEAKPTKGRRAYKELKNRRKKVRGKEKAKCSGAAKK

ID: TGGT1_240660
Sequence: MHKVLDQSRKSPDVSKFKDHQHSSSISRLKPFASDTGLRRFRSSISQKRRFQMGHPQIASGGPRGPSDSMKRGKFFRGKVNGVTKQDAADKKVRNKTAKAKARDLRRLLQKKREELPPEVVSRIEEEITSLDSVALHQKKNEKRKRFLVKRKQLYDKIKFYEGQKVRRKIKAERRALSELLRRRRDATNAEVDACTLDASIEEHQSNLRKHLDDLNYILRYPADEPYVALFPSGGPGSEETEKKREEMRVKIRQQILQERIEDENEDNAEGEEEDAFLVAEQPSGPQHRTSVMFDSDGSVVAEDEEEKNLEDVFAEAPDWVESPDDGTAGRGGRRTRGSSRGGFSDRKRSYHEHQHGVREAGSHGRGGAHTDSRGRQKTSSHRGQKHSATLDGGRGSTGTRSGSRGRAGVPGSLEKREESKEIRERKGGNSSQGTDKKQTQSLKEGRERREGEGKRPAKQIEQIGLSKRFAGGKHTVFESDSD

ID: TGGT1_248290
Sequence: MFVERLLHVHHGGLPITSLDFQ

### Download Sequences of Unknown Proteins for `ngphylogeny.fr`

I will now save the sequences of the identified unknown/hypothetical proteins into a FASTA formatted text file. This format is widely accepted by phylogenetic analysis tools like `ngphylogeny.fr`.

In [ ]:
# @title Default title text
fasta_output_file = "/content/drive/MyDrive/Colab Notebooks/tgg1/unknown_tgg1_proteins_fasta.txt"

with open(fasta_output_file, "w") as f:
    for index, row in unknown_proteins_df.iterrows():
        f.write(f">{row['clean_gene_id']}\n{row['sequence']}\n")

print(f"Successfully saved {len(unknown_proteins_df)} unknown protein sequences to {fasta_output_file}")
print("You can now download this file from your Google Drive or use it with tools like ngphylogeny.fr")

Successfully saved 478 unknown protein sequences to /content/drive/MyDrive/Colab Notebooks/tgg1/unknown_tgg1_proteins_fasta.txt
You can now download this file from your Google Drive or use it with tools like ngphylogeny.fr


In [ ]:
import pandas as pd

# Load the classified proteins DataFrame if not already in memory
# This assumes 'unknown_proteins_df' was created in a previous cell (e.g., cell 86a063a0)
# If running this cell independently, ensure 'out' and 'classification_column' are defined

# For robustness, re-load and filter if 'unknown_proteins_df' might not be in scope
# (though in a sequential run it should be)
output_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_tgg1_proteins.csv"
out = pd.read_csv(output_csv)

if 'category_y' in out.columns:
    classification_column = 'category_y'
elif 'category_x' in out.columns:
    classification_column = 'category_x'
else:
    raise KeyError("Neither 'category_y' nor 'category_x' found in the DataFrame. Cannot classify proteins.")

unknown_proteins_df = out[out[classification_column] == 'unknown_hypothetical']

print("Example FASTA formatted sequences:")
# Display the sequences of these proteins in the requested format
for index, row in unknown_proteins_df.head(5).iterrows():
    print(f">{row['clean_gene_id']}\n{row['sequence']}\n")

Example FASTA formatted sequences:
>TGGT1_215460
MERSKQSSRFFLCMQRPPPHSAHTPRGDFPVYAANNRMHARLFIRPSRLGSTLYFPAWQEERLRIRLQSAPSAEVKDPETFPSRFFRPSKRTTDGCTEWGTEAHSGFLSRHLLPRPGLLSPRSPLFSVTESNMADSGAFSLRFRKLKTNPLLQRKQFGVDILHPSRGSVSKKELVEKIAKQFRVSDSQSIVVFGLKTAFGGGRSSGFAMIYDNPNAAKKFENRFRLVRLGYVEAKPTKGRRAYKELKNRRKKVRGKEKAKCSGAAKK

>TGGT1_240660
MHKVLDQSRKSPDVSKFKDHQHSSSISRLKPFASDTGLRRFRSSISQKRRFQMGHPQIASGGPRGPSDSMKRGKFFRGKVNGVTKQDAADKKVRNKTAKAKARDLRRLLQKKREELPPEVVSRIEEEITSLDSVALHQKKNEKRKRFLVKRKQLYDKIKFYEGQKVRRKIKAERRALSELLRRRRDATNAEVDACTLDASIEEHQSNLRKHLDDLNYILRYPADEPYVALFPSGGPGSEETEKKREEMRVKIRQQILQERIEDENEDNAEGEEEDAFLVAEQPSGPQHRTSVMFDSDGSVVAEDEEEKNLEDVFAEAPDWVESPDDGTAGRGGRRTRGSSRGGFSDRKRSYHEHQHGVREAGSHGRGGAHTDSRGRQKTSSHRGQKHSATLDGGRGSTGTRSGSRGRAGVPGSLEKREESKEIRERKGGNSSQGTDKKQTQSLKEGRERREGEGKRPAKQIEQIGLSKRFAGGKHTVFESDSD

>TGGT1_248290
MFVERLLHVHHGGLPITSLDFQPHAAPWNMERLATAAANCVKIWVLHSHSPSVCRALLCGKSSEDAAPSQKRQKLQAAASGGTAAGKDSQTTGLDGEDRPPSATASVGASPGEPPRLSRPPLTVATCVWAASEHRSEIMTLRWAPTGEFLATCDADGNLFVYALNLPLP

#Old version

In [ ]:
from Bio import Entrez
import pandas as pd
import time
import re

# -----------------------------
# Settings
# -----------------------------
Entrez.email = "nickykundy@gmail.com"  # Please change this to your email

input_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/cleaned_tgg1_ids.csv"
output_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_by_ncbi.csv"

id_column = "clean_gene_id"  # change if your column name is different

unknown_keywords = [
    "hypothetical",
    "uncharacterized",
    "unknown",
    "unknown function",
    "protein of unknown function",
    "predicted protein",
]

def classify_product(product_name):
    name = str(product_name).lower().strip()

    if name == "":
        return "unknown_hypothetical"

    if any(k in name for k in unknown_keywords):
        return "unknown_hypothetical"

    return "known"


def search_ncbi_protein(clean_id):
    """
    Search NCBI Protein using clean TGME49 ID.
    Returns accession and protein title/product name if found.
    """

    # Toxoplasma gondii organism filter + clean ID
    term = f'"{clean_id}" AND "Toxoplasma gondii"[Organism]'

    handle = Entrez.esearch(
        db="protein",
        term=term,
        retmax=5
    )
    record = Entrez.read(handle)
    handle.close()

    ids = record["IdList"]

    if len(ids) == 0:
        return None

    # Read summaries of top hits
    handle = Entrez.esummary(
        db="protein",
        id=",".join(ids)
    )
    summaries = Entrez.read(handle)
    handle.close()

    # Use first hit
    first = summaries[0]

    accession = first.get("AccessionVersion", "")
    title = first.get("Title", "")

    return {
        "ncbi_accession": accession,
        "protein_title": title,
        "ncbi_uid": ids[0],
    }


# -----------------------------
# Main
# -----------------------------
df = pd.read_csv(input_csv)

if id_column not in df.columns:
    raise ValueError(f"Column '{id_column}' not found. Available columns: {list(df.columns)}")

results = []

unique_ids = df[id_column].dropna().astype(str).unique()

for i, clean_id in enumerate(unique_ids, start=1):
    print(f"[{i}/{len(unique_ids)}] Searching {clean_id}")

    try:
        result = search_ncbi_protein(clean_id)
    except Exception as e:
        print(f"  Failed: {clean_id} | {e}")
        result = None

    if result is None:
        row = {
            id_column: clean_id,
            "ncbi_accession": "",
            "protein_title": "",
            "ncbi_uid": "",
            "annotation_source": "not_found_in_ncbi",
            "category": "unknown_hypothetical",
        }
    else:
        protein_title = result["protein_title"]

        row = {
            id_column: clean_id,
            "ncbi_accession": result["ncbi_accession"],
            "protein_title": protein_title,
            "ncbi_uid": result["ncbi_uid"],
            "annotation_source": "ncbi",
            "category": classify_product(protein_title),
        }

    results.append(row)

    # Be gentle to NCBI server
    time.sleep(0.35)

anno = pd.DataFrame(results)

out = df.merge(anno, on=id_column, how="left")

out.to_csv(output_csv, index=False)

print("\nClassification summary:")
print(out["category"].value_counts(dropna=False))

print(f"\nSaved: {output_csv}")

[1/476] Searching TGGT1_215460
[2/476] Searching TGGT1_240660
[3/476] Searching TGGT1_248290
[4/476] Searching TGGT1_257780
[5/476] Searching TGGT1_259840
[6/476] Searching TGGT1_260600
[7/476] Searching TGGT1_270270
[8/476] Searching TGGT1_280390
[9/476] Searching TGGT1_285660
[10/476] Searching TGGT1_290630
[11/476] Searching TGGT1_293900
[12/476] Searching TGGT1_297730
[13/476] Searching TGGT1_203620
[14/476] Searching TGGT1_217520
[15/476] Searching TGGT1_223790
[16/476] Searching TGGT1_246920
[17/476] Searching TGGT1_275630
[18/476] Searching TGGT1_277840
[19/476] Searching TGGT1_281430
[20/476] Searching TGGT1_306220
[21/476] Searching TGGT1_202870
[22/476] Searching TGGT1_208050
[23/476] Searching TGGT1_227948
[24/476] Searching TGGT1_242020
[25/476] Searching TGGT1_259940
[26/476] Searching TGGT1_266310
[27/476] Searching TGGT1_312580
[28/476] Searching TGGT1_208400
[29/476] Searching TGGT1_224600
[30/476] Searching TGGT1_244700
[31/476] Searching TGGT1_289490
[32/476] Searchin

In [ ]:
import pandas as pd

# Load the classified proteins DataFrame
input_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_tgg1_proteins.csv"
out = pd.read_csv(input_csv)

# Use 'category_x' for classification, as it corresponds to the NCBI classification
# which resulted in 201 unknown_hypothetical proteins as mentioned by the user.
classification_column = 'category_x'

# Filter for unknown/hypothetical proteins
unknown_hypothetical_df = out[out[classification_column] == 'unknown_hypothetical']

# Define the output path for the new CSV file
output_unknown_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_tgg1_proteins.csv"

# Save the filtered DataFrame to a new CSV file
unknown_hypothetical_df.to_csv(output_unknown_csv, index=False)

print(f"Successfully saved {len(unknown_hypothetical_df)} unknown/hypothetical proteins to {output_unknown_csv}")
print("Head of the new DataFrame:")
print(unknown_hypothetical_df.head())

Successfully saved 200 unknown/hypothetical proteins to /content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_tgg1_proteins.csv
Head of the new DataFrame:
           original_id clean_gene_id  length  \
1   TGGT1_240660-t26_1  TGGT1_240660     483   
3   TGGT1_257780-t26_1  TGGT1_257780    1698   
4   TGGT1_259840-t26_1  TGGT1_259840    2091   
6   TGGT1_270270-t26_1  TGGT1_270270     377   
8  TGGT1_285660B-t26_1  TGGT1_285660    1196   

                                            sequence ncbi_accession  \
1  MHKVLDQSRKSPDVSKFKDHQHSSSISRLKPFASDTGLRRFRSSIS...     EPR60527.1   
3  MRSPFGPLRGWRRDLGSVLLSSDCPVVSPRPTSDGGFPTLLPGPSG...     EPR63044.1   
4  MVGQDFYYLCRHPNVFRVPTRDEPLLTHALPLEILDPPEAAPFSKR...     EPR62756.1   
6  MSSQASARAGLSPAEEAFLSYDWAKSAKWQAYLDSLYPTPPLNKMM...     EPR64744.1   
8  QRCEASGARRRRFLRRTFGILDEDNSADNNSLNIASPVVAEPSSST...            NaN   

                                       protein_title     ncbi_uid  \
1  hypothetical protein TGGT1_240660 [Toxoplasma 

In [ ]:
import pandas as pd

# Ensure unknown_hypothetical_df is available. If running this cell independently,
# you would need to load 'classified_tgg1_proteins.csv' and filter it first.
# Assuming it's available from previous execution (cell 2vJpRIWuCVrb).

# Define the output FASTA file path
fasta_blast_output_file = "/content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_for_blast.fasta"

with open(fasta_blast_output_file, "w") as f:
    for index, row in unknown_hypothetical_df.iterrows():
        # Write the sequence ID (clean_gene_id) as the FASTA header
        f.write(f">{row['clean_gene_id']}\n")
        # Write the protein sequence
        f.write(f"{row['sequence']}\n")

print(f"Successfully saved {len(unknown_hypothetical_df)} unknown/hypothetical protein sequences to {fasta_blast_output_file}")
print("This file is formatted for BLAST and other sequence analysis tools.")

Successfully saved 200 unknown/hypothetical protein sequences to /content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_for_blast.fasta
This file is formatted for BLAST and other sequence analysis tools.


In [ ]:
import pandas as pd

# Assuming unique_unknown_proteins_df is available from previous execution.
# For robustness, if running this cell independently, ensure it's loaded:
# input_classified_csv = "/content/drive/MyDrive/Colab Notebooks/tgg1/classified_tgg1_proteins.csv"
# out = pd.read_csv(input_classified_csv)
# classification_column = 'category_x' # or 'category_y'
# unknown_hypothetical_df = out[out[classification_column] == 'unknown_hypothetical'].copy()
# unique_unknown_proteins_df = unknown_hypothetical_df.drop_duplicates(subset=['sequence'])

# The kernel state shows unique_unknown_proteins_df has 200 rows, so we can split directly.

# Split into two files, each containing 100 sequences
num_proteins = len(unique_unknown_proteins_df)

if num_proteins < 200:
    print(f"Warning: Only {num_proteins} unique sequences found, which is less than 200 required for two files of 100 each. Splitting as best as possible.")
    split_point = num_proteins // 2
else:
    split_point = 100 # Take the first 100 for the first file, and the next 100 for the second

file1_df = unique_unknown_proteins_df.iloc[:split_point]
file2_df = unique_unknown_proteins_df.iloc[split_point:min(split_point + 100, num_proteins)]

# Define output file paths
fasta_file1_path = "/content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_blast_part1.fasta"
fasta_file2_path = "/content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_blast_part2.fasta"

# Function to write DataFrame to FASTA file
def write_df_to_fasta(df, filepath):
    with open(filepath, "w") as f:
        for index, row in df.iterrows():
            f.write(f">{row['clean_gene_id']}\n")
            f.write(f"{row['sequence']}\n")
    print(f"Saved {len(df)} sequences to {filepath}")

# Write to files
write_df_to_fasta(file1_df, fasta_file1_path)
write_df_to_fasta(file2_df, fasta_file2_path)

print("Splitting into FASTA files complete.")

Saved 100 sequences to /content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_blast_part1.fasta
Saved 100 sequences to /content/drive/MyDrive/Colab Notebooks/tgg1/unknown_hypothetical_proteins_blast_part2.fasta
Splitting into FASTA files complete.
